# Bayesian optimisation evidence
This notebook consolidates five measured baselines and renders charts from the completed Extra Trees study. The original MLP experiment used a different feature set and metric. The new comparable MLP run completed after the Extra Trees study and confirmed its lead. No new search is run here.

In [1]:
from pathlib import Path
import csv, json, os
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
evidence = Path("reports/bayesian")
local = Path("results/bayesian_extra_trees_01")
baseline = json.loads((evidence / "baseline_runs.json").read_text())
trials = json.loads((evidence / "trials.json").read_text())
comparison = json.loads((evidence / "comparison.json").read_text())
names = {
    "logistic": ("Logistic", "Linear", "C=1, LBFGS, fold-fitted scaling", 212),
    "hist_gradient_boosting": ("Hist. Boosting", "Boosting", "150 max iterations, 31 leaves, learning rate 0.1, L2=1", 212),
    "random_forest": ("Random Forest", "Bagged trees", "150 trees, depth 24, leaf 2, sqrt features", 212),
    "extra_trees": ("Extra Trees", "Randomised trees", "150 trees, depth 24, leaf 2, sqrt features", 212),
    "mlp": ("MLP", "Neural network", "64/32 hidden units, batch 256, 25 epochs, fold-fitted scaling", 212),
}
rows = []
for model, result in baseline.items():
    records = result["rec"]
    folds = [records[f"fold_{i}"] for i in range(5)]
    assert [r["metrics"]["all_points"]["observations"] for r in folds] == [28700, 40694, 49146, 62314, 27563]
    metrics = records["student_test"]["metrics"]
    test = metrics["all_points"]
    assert test["observations"] == 19160
    fold_metrics = [r["metrics"]["all_points"] for r in folds]
    name, family, cfg, seed = names[model]
    fit = [r.get("fit_seconds") for r in [*folds, records["student_test"]]]
    rows.append(dict(
        model=name, family=family, baseline_configuration=cfg, seed=seed,
        cv_top1_mean=float(np.mean([m["top1_accuracy"] for m in fold_metrics])),
        cv_macro_f1_mean=float(np.mean([m["macro_f1"] for m in fold_metrics])),
        cv_macro_f1_population_sd=float(np.std([m["macro_f1"] for m in fold_metrics])),
        student_top1=test["top1_accuracy"], student_macro_f1=test["macro_f1"],
        student_macro_precision=test["macro_precision"], student_macro_recall=test["macro_recall"],
        candidate_miss_rate=test["candidate_miss_rate"],
        categorical_log_loss=test["categorical_log_loss_with_miss_state"],
        candidate_binary_log_loss=metrics.get("candidate_binary_log_loss"),
        fitting_seconds=sum(fit) if all(v is not None for v in fit) else None,
        student_inference_ms_per_obs=records["student_test"].get("inference_ms_per_observation"),
        reliable_coverage=metrics.get("reliable_coverage"),
        reliable_top1=(metrics.get("reliable_points") or {}).get("top1_accuracy"),
        reliable_macro_f1=(metrics.get("reliable_points") or {}).get("macro_f1"),
        warnings=" | ".join(sorted({w for r in records.values() for w in r.get("warnings", [])})),
    ))
rows.sort(key=lambda r:r["cv_macro_f1_mean"], reverse=True)
with (evidence / "baseline_comparison.csv").open("w", newline="") as f:
    writer=csv.DictWriter(f, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)
print("Five comparable baselines ranked by mean arc Macro F1")
for r in rows:
    print(f'{r["model"]:16} {r["cv_macro_f1_mean"]:.5f} ± {r["cv_macro_f1_population_sd"]:.5f}  test Top-1={r["student_top1"]:.4%}  test F1={r["student_macro_f1"]:.5f}')



Five comparable baselines ranked by mean arc Macro F1
Extra Trees      0.65852 ± 0.04434  test Top-1=81.1065%  test F1=0.63245
Random Forest    0.64811 ± 0.04602  test Top-1=80.9656%  test F1=0.61396
Hist. Boosting   0.64705 ± 0.04413  test Top-1=80.7829%  test F1=0.61134
MLP              0.64321 ± 0.04862  test Top-1=79.4468%  test F1=0.60253
Logistic         0.64281 ± 0.04858  test Top-1=80.7777%  test F1=0.62856


In [2]:
plot_dir = evidence / "plots"
plot_dir.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 125, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.2, "axes.axisbelow": True})
colors = ["#35668f", "#007d68"]
def save(fig, name):
    fig.tight_layout()
    fig.savefig(plot_dir / f"{name}.png")
    plt.close(fig)
    print(plot_dir / f"{name}.png")
fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.barh([r["model"] for r in reversed(rows)], [r["cv_macro_f1_mean"] for r in reversed(rows)],
        xerr=[r["cv_macro_f1_population_sd"] for r in reversed(rows)], color=colors[0], capsize=3)
ax.set(xlabel="Mean five-fold arc Macro F1 ± population SD", title="Five-model baseline comparison")
save(fig, "01_baselines")
trial_no = [t["number"] for t in trials]
scores = [t["mean_macro_f1"] for t in trials]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(trial_no, scores, "o-", label="Completed trial", color=colors[0])
ax.step(trial_no, np.maximum.accumulate(scores), where="post", label="Running best", color=colors[1])
ax.set(xlabel="Trial", ylabel="Mean CV Macro F1", title="Bayesian search history", xticks=trial_no)
ax.legend()
save(fig, "02_trial_history")
# Six observations and four changing parameters do not support reliable global importance.
fig, axs = plt.subplots(2, 2, figsize=(8, 5), sharey=True)
for ax, key in zip(axs.flat, ["n_estimators","max_depth","min_samples_leaf","max_features"]):
    ax.scatter([t["point"][["n_estimators","max_depth","min_samples_leaf","max_features"].index(key)] for t in trials], scores, color=colors[0])
    ax.set(xlabel=key, ylabel="CV Macro F1")
fig.suptitle("Observed parameter relationships (parameters vary jointly)")
save(fig, "04_parameter_relationships")

pair = [trials[0], max(trials, key=lambda t:t["mean_macro_f1"])]
labels = ["Untuned", "Tuned"]
def mean_fold(trial, key):return float(np.mean([f["all_points"][key] for f in trial["folds"]]))
fig, axs = plt.subplots(1, 3, figsize=(8, 3))
for ax, key, label in zip(axs, ["macro_f1", "top1_accuracy","categorical_log_loss_with_miss_state"], ["Macro F1 ↑","Top-1 ↑","Categorical log loss ↓"]):
    values = [mean_fold(t,key) for t in pair]
    ax.bar(labels, values, color=colors)
    ax.set_title(label)
    for i,v in enumerate(values):ax.text(i,v,f"{v:.4f}",ha="center",va="bottom",fontsize=8)
    ax.set_ylim(0,max(values)*1.15)
fig.suptitle("Five-fold CV comparison")
save(fig, "05_cv_comparison")
fig, axs = plt.subplots(1, 2, figsize=(6, 3))
for ax, key, label in zip(axs, ["top1_accuracy","macro_f1"], ["Top-1","Arc Macro F1"]):
    values = [comparison[k]["all_points"][key] for k in ["untuned","tuned"]]
    ax.bar(labels, values, color=colors)
    ax.set_ylim(0,1)
    ax.set_title(label)
    for i,v in enumerate(values):ax.text(i,v,f"{v:.4f}",ha="center",va="bottom",fontsize=8)
fig.suptitle("Student-test comparison")
save(fig, "06_student_test")
fig, ax = plt.subplots(figsize=(6, 3))
for i, t in enumerate(pair):
    ax.plot(range(1,6),[f["all_points"]["macro_f1"] for f in t["folds"]],"o-",color=colors[i],label=labels[i])
ax.set(xlabel="Fold", ylabel="Arc Macro F1", xticks=range(1,6), title="Held-out fold scores")
ax.legend()
save(fig, "07_fold_scores")
fig, (ax, bx) = plt.subplots(1, 2, figsize=(8, 3))
ax.bar(trial_no,[t["wall_seconds"] for t in trials],color=colors[0])
ax.set(xlabel="Trial",ylabel="Seconds",title="Five-fold wall time")
bx.bar(labels,[comparison[k]["inference_ms_per_observation"] for k in ["untuned","tuned"]],color=colors)
bx.set(ylabel="ms per observation",title="Student-test scoring")
save(fig, "08_compute")
print("No global hyperparameter importance estimated from six trials.")


reports/bayesian/plots/01_baselines.png
reports/bayesian/plots/02_trial_history.png
reports/bayesian/plots/04_parameter_relationships.png
reports/bayesian/plots/05_cv_comparison.png
reports/bayesian/plots/06_student_test.png
reports/bayesian/plots/07_fold_scores.png
reports/bayesian/plots/08_compute.png
No global hyperparameter importance estimated from six trials.
